# Modulo 6 · Lezione 5 — Inferenza statistica

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Popolazione vs campione |
| 2 | Distribuzione campionaria ed errore standard |
| 3 | Teorema del Limite Centrale |
| 4 | Intervalli di confidenza |
| 5 | Test di ipotesi (t-test) |
| 6 | Errori di tipo I e II |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Librerie pronte")

---
## Motivazione

```
Quanto tempo impiega in media il mio modello a fare una previsione?
Questo nuovo modello è significativamente migliore del precedente?
La differenza che vedo è reale o può essere dovuta al caso?

La statistica inferenziale risponde a queste domande partendo da dati limitati.

Non possiamo mai osservare TUTTO → inferiamo sulla popolazione
usando ciò che osserviamo in un campione.
```

---
## 1. Popolazione vs campione

In [ ]:
# Simuliamo la "popolazione reale" (normalmente inaccessibile)
np.random.seed(42)
POBLACION_TIEMPOS = np.random.gamma(shape=2, scale=50, size=100_000)  # ms di latenza

mu_real    = POBLACION_TIEMPOS.mean()
sigma_real = POBLACION_TIEMPOS.std(ddof=0)

print("=== Parametri REALI della popolazione ===")
print(f"μ (media reale)     = {mu_real:.4f} ms")
print(f"σ (dev. std reale)  = {sigma_real:.4f} ms")
print(f"Dimensione popolazione: {len(POBLACION_TIEMPOS):,}")

# Prendiamo campioni di dimensione n
print("\n=== Stime da campioni ===")
print(f"{'n':>6} {'media_campione':>15} {'errore':>10} {'std_campione':>13}")

for n in [5, 10, 30, 100, 1000]:
    muestra   = np.random.choice(POBLACION_TIEMPOS, size=n, replace=False)
    media_m   = muestra.mean()
    std_m     = muestra.std(ddof=1)
    error     = abs(media_m - mu_real)
    print(f"{n:>6} {media_m:>15.4f} {error:>10.4f} {std_m:>13.4f}")

print(f"\n→ Più n è grande, più le stime si avvicinano ai parametri reali")

---
## 2. Distribuzione campionaria

```
Se prendiamo molti campioni e calcoliamo la media di ciascuno,
quelle medie formano la "distribuzione campionaria della media".

Media della distribuzione campionaria = μ (media della popolazione)
Dev. standard della distribuzione campionaria = Errore Standard = σ / √n

Più n è grande → minore l'errore standard → stime più precise
```

In [ ]:
n_muestras = 1000
fig, axes  = plt.subplots(1, 3, figsize=(14, 4))

for ax, n in zip(axes, [5, 30, 100]):
    medias = [np.random.choice(POBLACION_TIEMPOS, size=n).mean() for _ in range(n_muestras)]
    medias = np.array(medias)

    error_std_teorico = sigma_real / np.sqrt(n)
    error_std_empir   = medias.std()

    ax.hist(medias, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    # Curva normale teorica
    x_plot = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x_plot, stats.norm.pdf(x_plot, mu_real, error_std_teorico),
            'r-', lw=2, label='Normale teorica')
    ax.axvline(mu_real, color='orange', ls='--', lw=2, label=f'μ={mu_real:.1f}')
    ax.set_title(f'n={n}\nEE teorico={error_std_teorico:.2f}, empirico={error_std_empir:.2f}')
    ax.legend(fontsize=7)
    ax.set_xlabel('Media campionaria (ms)')

plt.suptitle('Distribuzione campionaria della media (1000 campioni)', fontsize=11)
plt.tight_layout(); plt.show()

print("Osservazioni:")
print("  1. Le medie campionarie si distribuiscono attorno alla μ reale")
print("  2. Più n è grande, più stretta è la distribuzione (maggiore precisione)")
print("  3. Anche se la popolazione è asimmetrica (gamma), le medie sono normali")

---
## 3. Teorema del Limite Centrale (TLC)

```
Indipendentemente dalla forma della distribuzione originale,
se n è sufficientemente grande (≥30),
la distribuzione campionaria della media si approssima a una normale.

→ È ciò che rende possibile l'inferenza statistica nella pratica.
→ In IA: giustifica assumere la normalità nei residui, negli errori di previsione, ecc.
```

In [ ]:
# Dimostrare il TLC con 3 distribuzioni molto diverse
np.random.seed(42)
distribuciones = [
    ('Uniforme', lambda n: np.random.uniform(0, 1, n)),
    ('Esponenziale', lambda n: np.random.exponential(1, n)),
    ('Bernoulli p=0.1', lambda n: np.random.binomial(1, 0.1, n)),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for row, (nombre, generador) in enumerate(distribuciones):
    # Popolazione
    pob = generador(10000)
    axes[row, 0].hist(pob, bins=40, density=True, alpha=0.7, color='coral')
    axes[row, 0].set_title(f'{nombre}\n(distribuzione originale)')

    # Distribuzioni campionarie per n=2, 10, 30
    for col, n in enumerate([2, 10, 30]):
        medias = [generador(n).mean() for _ in range(2000)]
        medias = np.array(medias)

        axes[row, col+1].hist(medias, bins=40, density=True, alpha=0.7,
                              color='steelblue', edgecolor='white')

        # Overlay normale teorica
        mu_t  = medias.mean()
        std_t = medias.std()
        x_n   = np.linspace(medias.min(), medias.max(), 200)
        axes[row, col+1].plot(x_n, stats.norm.pdf(x_n, mu_t, std_t), 'r-', lw=2)
        skew_m = stats.skew(medias)
        axes[row, col+1].set_title(f'n={n}\nskewness={skew_m:.3f}')

plt.suptitle('Teorema del Limite Centrale', fontsize=13)
plt.tight_layout(); plt.show()

print("Per n=30, la distribuzione delle medie è quasi normale in tutti e tre i casi.")

---
## 4. Intervalli di confidenza

```
"Con il 95% di confidenza, la media reale è compresa tra [a, b]"

IC al 95% = x̄ ± t* × (s/√n)

  x̄  = media campionaria
  s   = deviazione standard campionaria
  n   = dimensione del campione
  t*  = valore critico della t di Student (≈1.96 per n grande, 95%)

Interpretazione CORRETTA:
  Se ripetessi l'esperimento 100 volte, circa 95 degli
  intervalli conterrebbero la media reale.

Non significa: "c'è il 95% di probabilità che μ sia in [a, b]"
```

In [ ]:
def intervalo_confianza(datos, confianza=0.95):
    n    = len(datos)
    mean = np.mean(datos)
    se   = stats.sem(datos)  # errore standard = s / sqrt(n)
    ic   = stats.t.interval(confianza, df=n-1, loc=mean, scale=se)
    return mean, ic[0], ic[1], se

# Un campione
np.random.seed(42)
muestra = np.random.choice(POBLACION_TIEMPOS, size=30)
mean, ic_inf, ic_sup, se = intervalo_confianza(muestra)

print(f"Campione n=30:")
print(f"  Media campionaria: {mean:.4f} ms")
print(f"  Errore standard:   {se:.4f} ms")
print(f"  IC 95%:            [{ic_inf:.4f}, {ic_sup:.4f}] ms")
print(f"  μ reale:           {mu_real:.4f} ms → {'Dentro ✓' if ic_inf <= mu_real <= ic_sup else 'Fuori ✗'}")

# Effetto del livello di confidenza
print("\nIC per lo stesso campione, diversi livelli:")
for conf in [0.80, 0.90, 0.95, 0.99]:
    m, a, b, _ = intervalo_confianza(muestra, conf)
    print(f"  {int(conf*100)}%: [{a:.2f}, {b:.2f}]  (ampiezza={b-a:.2f})")

# Visualizzare 100 IC
np.random.seed(0)
n_experimentos = 100
exitos = 0
fig, ax = plt.subplots(figsize=(12, 6))

for i in range(n_experimentos):
    m_i = np.random.choice(POBLACION_TIEMPOS, size=30)
    mean_i, a_i, b_i, _ = intervalo_confianza(m_i)
    dentro = a_i <= mu_real <= b_i
    if dentro: exitos += 1
    color = 'steelblue' if dentro else 'red'
    ax.plot([a_i, b_i], [i, i], color=color, alpha=0.5, lw=1)
    ax.plot(mean_i, i, 'o', color=color, ms=2)

ax.axvline(mu_real, color='orange', ls='--', lw=2, label=f'μ reale = {mu_real:.1f}')
ax.set_xlabel('Latenza (ms)')
ax.set_title(f'100 intervalli di confidenza al 95% — Contengono μ: {exitos}/100')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n{exitos} IC su 100 contengono la media reale (atteso ≈ 95)")

---
## 5. Test di ipotesi

```
Vogliamo sapere se una differenza osservata è reale o frutto del caso.

H₀ (ipotesi nulla):        non c'è differenza / effetto
H₁ (ipotesi alternativa):  c'è differenza / effetto

p-value: probabilità di ottenere questo risultato (o più estremo)
         se H₀ fosse vera.

  p < α (es. 0.05)  → rifiutiamo H₀ → differenza significativa
  p ≥ α             → non possiamo rifiutare H₀

ATTENZIONE! "Non rifiutare H₀" ≠ "H₀ è vera"
            "Significativo" ≠ "importante" o "grande"
```

In [ ]:
# Scenario: confrontare la latenza di due versioni del modello
np.random.seed(42)
version_A = np.random.normal(100, 15, 30)  # ms
version_B = np.random.normal(95, 18, 30)   # ms (B è davvero più veloce?)

print(f"Versione A: media={version_A.mean():.2f} ms, std={version_A.std():.2f}")
print(f"Versione B: media={version_B.mean():.2f} ms, std={version_B.std():.2f}")
print(f"Differenza osservata: {version_A.mean() - version_B.mean():.2f} ms")

# t-test indipendente (Welch)
t_stat, p_valor = stats.ttest_ind(version_A, version_B, equal_var=False)
alpha = 0.05

print(f"\nt-statistica:  {t_stat:.4f}")
print(f"p-value:       {p_valor:.4f}")
print(f"Alpha:         {alpha}")
print(f"Decisione: {'Rifiutiamo H₀ — differenza significativa' if p_valor < alpha else 'Non rifiutiamo H₀ — differenza non significativa'}")

# t-test a un campione (la media è diversa da un valore?)
print("\n=== La latenza di B supera la soglia di 90ms? ===")
t2, p2 = stats.ttest_1samp(version_B, popmean=90)
print(f"H₀: μ_B = 90ms")
print(f"t={t2:.4f}, p={p2:.4f}")
print(f"Decisione: {'Sì, supera significativamente 90ms' if p2 < alpha else 'No, non supera significativamente 90ms'}")

# t-test appaiato (stesso modello, prima e dopo ottimizzazione)
print("\n=== Test appaiato: prima e dopo l'ottimizzazione ===")
antes = np.random.normal(120, 10, 20)
despues = antes - np.random.normal(8, 5, 20)  # miglioramento medio di 8ms
t3, p3 = stats.ttest_rel(antes, despues)
print(f"Riduzione media: {(antes - despues).mean():.2f} ms")
print(f"t={t3:.4f}, p={p3:.4f}")
print(f"L'ottimizzazione {'è' if p3 < alpha else 'NON è'} statisticamente significativa")

---
## 6. Errori di tipo I e II

```
                    H₀ vera           H₀ falsa
Rifiutare H₀    ┃ Errore Tipo I (α) ┃ Corretto (Potenza) ┃
Non rifiutare H₀┃ Corretto          ┃ Errore Tipo II (β) ┃

Tipo I  (Falso Positivo): "Rileviamo qualcosa che non esiste"
         Probabilità = α (livello di significatività)

Tipo II (Falso Negativo): "Non rileviamo qualcosa che esiste"
         Probabilità = β

Potenza statistica = 1 - β = probabilità di rilevare un effetto reale
  Aumenta con: n maggiore, effetto maggiore, α maggiore, minore variabilità
```

In [ ]:
# Simulare errori di tipo I e tipo II
np.random.seed(42)
n_sim    = 2000
alpha    = 0.05

# Errore tipo I: H₀ vera (μ_A = μ_B = 100)
falsas_positivas = sum(
    1 for _ in range(n_sim)
    if stats.ttest_ind(np.random.normal(100, 15, 30),
                       np.random.normal(100, 15, 30))[1] < alpha
)
print(f"Errori Tipo I (atteso ~{alpha*100:.0f}%): {falsas_positivas/n_sim*100:.1f}%")

# Errore tipo II: H₀ falsa (μ_B = 93, differenza reale di 7ms)
for n in [10, 30, 100]:
    detectados = sum(
        1 for _ in range(n_sim)
        if stats.ttest_ind(np.random.normal(100, 15, n),
                           np.random.normal(93, 15, n))[1] < alpha
    )
    poder = detectados / n_sim
    tipo2 = 1 - poder
    print(f"n={n:3d}: Potenza={poder:.3f} ({poder*100:.1f}%), Errore Tipo II={tipo2:.3f}")

# Visualizzazione
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(80, 130, 400)

dist_H0 = stats.norm(100, 15/np.sqrt(30))
dist_H1 = stats.norm(93, 15/np.sqrt(30))
t_critico_val = dist_H0.ppf(alpha / 2)  # test bilaterale
t_critico_sup = dist_H0.ppf(1 - alpha / 2)

ax.plot(x, dist_H0.pdf(x), 'b-', lw=2, label='H₀: μ=100 (nessuna differenza)')
ax.plot(x, dist_H1.pdf(x), 'r-', lw=2, label='H₁: μ=93 (differenza reale)')

# Zona di rifiuto (Tipo I)
x_alpha = np.linspace(80, t_critico_val, 100)
ax.fill_between(x_alpha, dist_H0.pdf(x_alpha), alpha=0.3, color='blue', label='Errore Tipo I (α)')
x_alpha2 = np.linspace(t_critico_sup, 130, 100)
ax.fill_between(x_alpha2, dist_H0.pdf(x_alpha2), alpha=0.3, color='blue')

# Zona Tipo II (β)
x_beta = np.linspace(t_critico_val, t_critico_sup, 200)
ax.fill_between(x_beta, dist_H1.pdf(x_beta), alpha=0.3, color='red', label='Errore Tipo II (β)')

ax.axvline(t_critico_val, color='purple', ls='--', alpha=0.5)
ax.axvline(t_critico_sup, color='purple', ls='--', alpha=0.5, label='Valore critico (α=0.05)')
ax.set_xlabel('Media campionaria (ms)'); ax.set_ylabel('Densità')
ax.set_title('Errori Tipo I e Tipo II')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Intervallo di confidenza per metriche di modello

**Consegna:** Valuti il tuo modello su 40 esempi e ottieni queste accuracy per batch:

```python
np.random.seed(5)
accuracies = np.random.beta(8, 2, 40)  # ~media 0.80
```

1. Media e deviazione standard
2. IC al 90%, 95% e 99%
3. Puoi affermare con il 95% di confidenza che il modello supera il 75% di accuracy?
4. Quanti esempi ti servirebbero perché l'IC 95% abbia un'ampiezza massima di 0.05?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
from scipy import stats

np.random.seed(5)
accuracies = np.random.beta(8, 2, 40)

# 1.
media = accuracies.mean()
std   = accuracies.std(ddof=1)
print(f"Media: {media:.4f}  |  Std: {std:.4f}")

# 2.
for conf in [0.90, 0.95, 0.99]:
    ic = stats.t.interval(conf, df=len(accuracies)-1,
                          loc=media, scale=stats.sem(accuracies))
    ancho = ic[1] - ic[0]
    print(f"IC {int(conf*100)}%: [{ic[0]:.4f}, {ic[1]:.4f}]  ampiezza={ancho:.4f}")

# 3. IC 95%
ic95 = stats.t.interval(0.95, df=len(accuracies)-1,
                        loc=media, scale=stats.sem(accuracies))
print(f"\nSupera 0.75?")
print(f"  Limite inferiore IC 95%: {ic95[0]:.4f}")
print(f"  {'Sì, con il 95% di confidenza il modello supera 0.75' if ic95[0] > 0.75 else \"No, l'IC include valori ≤ 0.75\"}")

# 4. n necessario per ampiezza ≤ 0.05
# Ampiezza = 2 * z * (std / sqrt(n))  → n = (2 * z * std / ampiezza)²
z95 = 1.96  # per n grande
ancho_objetivo = 0.05
n_necesario = np.ceil((2 * z95 * std / ancho_objetivo) ** 2)
print(f"\nPer ampiezza ≤ 0.05: n ≥ {n_necesario:.0f} (ne hai {len(accuracies)})")
```
</details>

---
### Esercizio 2 — A/B test di due modelli

**Consegna:** Confronta due versioni di un classificatore:

```python
np.random.seed(1)
f1_v1 = np.random.normal(0.82, 0.04, 25)
f1_v2 = np.random.normal(0.85, 0.06, 25)
```

1. La differenza di F1 è statisticamente significativa? (α=0.05)
2. Ripeti con n=5 e n=200. Cambia la conclusione?
3. Con n=200, sarebbe significativa una differenza reale di solo lo 0.5%?
4. Cosa conta di più: il p-value o la dimensione dell'effetto?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
from scipy import stats

def test_ab(f1, f2, descripcion):
    t, p = stats.ttest_ind(f1, f2, equal_var=False)
    diff = f2.mean() - f1.mean()
    # d di Cohen (dimensione dell'effetto)
    s_pool = np.sqrt((f1.std(ddof=1)**2 + f2.std(ddof=1)**2) / 2)
    d = diff / s_pool
    sig = p < 0.05
    print(f"{descripcion:30} diff={diff:+.4f}  p={p:.4f}  d={d:.3f}  {'✓ sig' if sig else '✗ non sig'}")

np.random.seed(1)
# 1 e 2: diversi n
for n in [5, 25, 200]:
    v1 = np.random.normal(0.82, 0.04, n)
    v2 = np.random.normal(0.85, 0.06, n)
    test_ab(v1, v2, f"n={n}")

# 3: differenza molto piccola con n=200
np.random.seed(1)
v1_big = np.random.normal(0.820, 0.04, 200)
v2_big = np.random.normal(0.825, 0.04, 200)  # solo 0.5% di differenza
test_ab(v1_big, v2_big, "n=200, diff=0.005")

print()
print("4. Conclusione:")
print("   Con n grande, differenze MINIME sono statisticamente significative.")
print("   La d di Cohen misura la dimensione dell'effetto: d<0.2 trascurabile, 0.5 moderato, >0.8 grande.")
print("   Un p-value significativo non implica che la differenza sia praticamente rilevante.")
```
</details>

---
### Esercizio 3 — Campionamento stratificato

**Consegna:** Hai un dataset molto sbilanciato:

```python
np.random.seed(3)
clase_0 = np.random.normal(50, 10, 900)   # 90%
clase_1 = np.random.normal(70, 15, 100)   # 10%
```

1. Calcola la media globale del dataset completo
2. Prendi un campione casuale semplice (n=50). È rappresentativo?
3. Implementa un campionamento stratificato (proporzionale): 45 di classe_0, 5 di classe_1
4. Confronta gli IC di entrambi i metodi di campionamento rispetto alla media reale

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
from scipy import stats

np.random.seed(3)
clase_0 = np.random.normal(50, 10, 900)
clase_1 = np.random.normal(70, 15, 100)
dataset  = np.concatenate([clase_0, clase_1])

media_real = dataset.mean()
print(f"1. Media reale del dataset: {media_real:.4f}")

def ic95(muestra):
    ic = stats.t.interval(0.95, df=len(muestra)-1,
                          loc=muestra.mean(), scale=stats.sem(muestra))
    return muestra.mean(), ic[0], ic[1]

# 2. Campione casuale semplice
np.random.seed(42)
muestra_simple = np.random.choice(dataset, size=50, replace=False)
p_clase1_simple = np.sum(muestra_simple > 60) / 50  # proxy della classe_1
m, a, b = ic95(muestra_simple)
print(f"\n2. Campione semplice (n=50):")
print(f"   Media={m:.4f}  IC=[{a:.4f}, {b:.4f}]")
print(f"   Contiene la media reale? {'Sì' if a <= media_real <= b else 'No'}")

# 3. Campionamento stratificato
m0  = np.random.choice(clase_0, size=45, replace=False)
m1  = np.random.choice(clase_1, size=5,  replace=False)
muestra_estrat = np.concatenate([m0, m1])
m2, a2, b2 = ic95(muestra_estrat)
print(f"\n3. Campione stratificato (45+5=50):")
print(f"   Media={m2:.4f}  IC=[{a2:.4f}, {b2:.4f}]")
print(f"   Contiene la media reale? {'Sì' if a2 <= media_real <= b2 else 'No'}")

# 4. Confronto
print(f"\n4. Media reale:              {media_real:.4f}")
print(f"   Errore campione semplice:   {abs(m - media_real):.4f}")
print(f"   Errore campione stratif.:   {abs(m2 - media_real):.4f}")
print("   → Il campionamento stratificato garantisce la rappresentazione della classe minoritaria")
```
</details>

---
### Esercizio 4 — Validazione del modello con bootstrap

**Consegna:** Il bootstrap è una tecnica di ricampionamento per stimare qualsiasi statistica:

```
Per B ripetizioni:
  1. Prendi un campione con reinserimento (stessa dimensione dell'originale)
  2. Calcola la statistica
L'IC bootstrap = percentili [2.5, 97.5] delle B stime
```

Implementa il bootstrap e calcola l'IC 95% dell'F1-score di un classificatore usando:
```python
np.random.seed(7)
predicciones = np.random.binomial(1, 0.8, 200)  # modello con accuracy dell'80% circa
etiquetas    = np.random.binomial(1, 0.5, 200)   # classi bilanciate
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
predicciones = np.random.binomial(1, 0.8, 200)
etiquetas    = np.random.binomial(1, 0.5, 200)

def calcular_f1(pred, real):
    tp = np.sum((pred == 1) & (real == 1))
    fp = np.sum((pred == 1) & (real == 0))
    fn = np.sum((pred == 0) & (real == 1))
    if tp + fp == 0 or tp + fn == 0: return 0.0
    precision = tp / (tp + fp)
    recall    = tp / (tp + fn)
    if precision + recall == 0: return 0.0
    return 2 * precision * recall / (precision + recall)

f1_original = calcular_f1(predicciones, etiquetas)
print(f"F1 sui dati originali: {f1_original:.4f}")

# Bootstrap
B = 2000
n = len(predicciones)
f1_boot = np.zeros(B)

for i in range(B):
    idx        = np.random.randint(0, n, size=n)  # con reinserimento
    f1_boot[i] = calcular_f1(predicciones[idx], etiquetas[idx])

ic_inf = np.percentile(f1_boot, 2.5)
ic_sup = np.percentile(f1_boot, 97.5)

print(f"IC bootstrap 95%: [{ic_inf:.4f}, {ic_sup:.4f}]")
print(f"Bias stimato:     {f1_boot.mean() - f1_original:.4f}")

plt.figure(figsize=(8, 4))
plt.hist(f1_boot, bins=40, density=True, alpha=0.7, color='steelblue')
plt.axvline(f1_original, color='red',   ls='-',  lw=2, label=f'F1 originale={f1_original:.3f}')
plt.axvline(ic_inf,       color='orange', ls='--', lw=2, label=f'IC inferiore={ic_inf:.3f}')
plt.axvline(ic_sup,       color='orange', ls='--', lw=2, label=f'IC superiore={ic_sup:.3f}')
plt.xlabel('F1-score'); plt.title(f'Bootstrap F1 (B={B})')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 5 (sfida) — Motore di A/B testing

**Consegna:** Implementa una funzione `ab_test(A, B, alpha=0.05)` che:

1. Esegue il t-test, verifica la normalità (Shapiro) e l'omogeneità delle varianze (Levene)
2. Calcola la dimensione dell'effetto (d di Cohen)
3. Calcola la potenza statistica raggiunta (usando `stats.power`)
4. Restituisce un report con: differenza significativa?, magnitudine dell'effetto (trascurabile/piccolo/moderato/grande), raccomandazione

Provala con 3 scenari: nessuna differenza, differenza piccola (d≈0.2), differenza grande (d≈0.8).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
from scipy import stats

def ab_test(A, B, alpha=0.05):
    A, B = np.array(A), np.array(B)

    # Normalità
    _, p_norm_A = stats.shapiro(A)
    _, p_norm_B = stats.shapiro(B)
    normales    = p_norm_A > 0.05 and p_norm_B > 0.05

    # Varianze
    _, p_levene  = stats.levene(A, B)
    equal_var    = p_levene > 0.05

    # t-test
    t_stat, p_val = stats.ttest_ind(A, B, equal_var=equal_var)
    sig = p_val < alpha

    # d di Cohen
    diff   = B.mean() - A.mean()
    s_pool = np.sqrt((A.std(ddof=1)**2 + B.std(ddof=1)**2) / 2)
    d      = abs(diff / s_pool) if s_pool > 0 else 0

    if   d < 0.2: magnitud = "trascurabile"
    elif d < 0.5: magnitud = "piccolo"
    elif d < 0.8: magnitud = "moderato"
    else:         magnitud = "grande"

    # Report
    print("=" * 55)
    print(f"  A: n={len(A)}, μ={A.mean():.4f}, σ={A.std(ddof=1):.4f}")
    print(f"  B: n={len(B)}, μ={B.mean():.4f}, σ={B.std(ddof=1):.4f}")
    print(f"  Differenza B-A: {diff:+.4f}")
    print(f"  Normalità:         {'Sì' if normales else 'No (considera un test non parametrico)'}")
    print(f"  Varianze uguali:   {'Sì' if equal_var else 'No (correzione di Welch applicata)'}")
    print(f"  t={t_stat:.4f}  p={p_val:.4f}")
    print(f"  Significativo (α={alpha}): {'Sì ✓' if sig else 'No ✗'}")
    print(f"  d di Cohen: {d:.3f} → effetto {magnitud}")
    print(f"  Raccomandazione: ", end="")
    if sig and d >= 0.5:
        print("Adottare B (differenza significativa e rilevante)")
    elif sig and d < 0.2:
        print("Differenza significativa ma trascurabile. Ne vale il costo?")
    elif not sig:
        print("Non c'è evidenza sufficiente. Ampliare il campione o accettare l'uguaglianza.")
    else:
        print("Differenza moderata. Valutare costo/beneficio.")
    print()

np.random.seed(0)
ab_test(np.random.normal(5, 1, 50), np.random.normal(5, 1, 50))       # nessuna differenza
ab_test(np.random.normal(5, 1, 50), np.random.normal(5.2, 1, 50))     # d≈0.2
ab_test(np.random.normal(5, 1, 50), np.random.normal(5.8, 1, 50))     # d≈0.8
```
</details>

---
## Riepilogo

```python
from scipy import stats

# Errore standard
se = stats.sem(muestra)               # s / sqrt(n)

# Intervallo di confidenza
ic = stats.t.interval(0.95, df=n-1, loc=media, scale=se)

# Test di ipotesi
t, p = stats.ttest_1samp(x, popmean=val)    # 1 campione vs valore
t, p = stats.ttest_ind(x, y, equal_var=False)  # 2 campioni indipendenti
t, p = stats.ttest_rel(x, y)               # appaiato (prima/dopo)

# Diagnostica
_, p_shapiro = stats.shapiro(x)         # normalità
_, p_levene  = stats.levene(x, y)       # omogeneità delle varianze
skew         = stats.skew(x)            # asimmetria
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 6** introduciamo **pandas**, lo strumento indispensabile per l'analisi di dati reali: Series, DataFrame, filtraggio, raggruppamento e pulizia dei dati.